# B2.2 · Threat modelling from what the estate already knows

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.1 · Reading the repository: history, index, components, map](https://spbreed.github.io/cyber-commons/lessons/B2.1.html)**.

| | |
|---|---|
| Tools used | OWASP Threat Dragon, GLM-4.6, Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A threat model produced in a workshop describes the system as it was on the day of the workshop, and it is derived from the code alone — so two deployments of the same repository, one behind a private load balancer with no egress and one on the internet with a wildcard trust policy, get the same model. It is wrong about both.

> **At CyberTravels.** The threat model that said “CyberTravels answers questions” is still on file. Deriving it from the architecture on every release is what would have caught the refund endpoint appearing.

## 2 · The framework

```
   six static inputs, all already in the estate

   code analysis      what the code COULD reach     (stage 4)
   cloud policy       is it on the internet         (security groups, WAF)
   CSPM               is the bucket public TODAY
   entitlements       what the role may do
   IAM                who can become that role
   egress policy      can anything leave
          |
          v  derive, mechanically
   +---------------------------+
   | ranked threats, as data   |
   +---------------------------+
          |
        DIFFED against the last run - and the diff has to count
        ESCALATION, not only arrival, or a terraform-only pull
        request raises every score and passes the gate
```

Phase 2 opens with the stage everyone claims to do and almost nobody re-runs.

**Stage 5 — Threat modelling.** Derive, mechanically: high-value assets,
untrusted entry points, and the attack vectors that connect them.

The word doing the work is *mechanically*. A threat model produced by hand in a
workshop is a snapshot; it is stale the moment an entry point is added, and
adding an entry point is a Tuesday. A model **derived** is regenerated whenever
its inputs change, so the useful artefact is not the model — it is the **diff
between two models**.

### Derived from what, exactly

The architecture map from stage 4 is the first input and it is not sufficient.
It tells you what the code *could* reach. It says nothing about whether that
path is exposed, what identity walks it, or whether anything could leave at the
end of it — and those three questions are the difference between a finding and
a fire.

Every one of the answers is already written down somewhere in the estate, as
configuration, in a machine-readable file. The stage's job is to read all of it:

| Input | What only it can tell you |
|---|---|
| **Code analysis** — the stage-4 map | which sinks an entry point reaches |
| **CSPM findings** | that the bucket behind that sink is public, today |
| **Cloud security policy** — security groups, ingress, WAF | whether the entry point is reachable from the internet at all |
| **Entitlement and role access** | what the caller's role may do once it is through |
| **IAM** — trust policies, assume-role chains | who can become that role, and from where |
| **Egress policy** — NetworkPolicy, firewall rules | whether anything can leave once it is in |

Read only the code and you produce a threat model that is identical for two
deployments of the same repository, one of which is behind a private load
balancer with no egress and one of which is not. That model is wrong about both.

The output has to be data — ranked, machine-readable, diffable — because the
next stage prioritises against it and CI gates on the delta.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The six inputs, and what each one contributes

<div style="display:flex;gap:6px;align-items:stretch;flex-wrap:wrap;font-family:ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,sans-serif;margin:6px 0 2px"><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">code</div><div style="border:1px solid rgba(77,155,255,.55);border-left:3px solid #4D9BFF;border-radius:8px;background:rgba(77,155,255,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128269;</span> stage-4 map</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">entry points, flows, sinks, trust boundaries</div><div style="font-size:10px;color:#4D9BFF;margin-top:5px;font-weight:600;letter-spacing:.02em">WHAT COULD BE REACHED</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">exposure</div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#9729;&#65039;</span> cloud security policy</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">security groups, ingress, WAF — is this entry point on the internet</div></div><div style="border:1px solid rgba(224,92,75,.55);border-left:3px solid #E05C4B;border-radius:8px;background:rgba(224,92,75,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128680;</span> CSPM findings</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">the bucket behind that sink is public, today</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">identity</div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128273;</span> entitlement and roles</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">what the caller may do once it is through</div></div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128100;</span> IAM trust policy</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">who can assume that role, and from where</div><div style="font-size:10px;color:#E0912F;margin-top:5px;font-weight:600;letter-spacing:.02em">A2.3</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">exfiltration</div><div style="border:1px solid rgba(63,160,107,.55);border-left:3px solid #3FA06B;border-radius:8px;background:rgba(63,160,107,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128683;</span> egress policy</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">NetworkPolicy and firewall rules — can anything leave at the end of the path</div><div style="font-size:10px;color:#3FA06B;margin-top:5px;font-weight:600;letter-spacing:.02em">A3.2</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">output</div><div style="border:1px solid rgba(77,155,255,.55);border-left:3px solid #4D9BFF;border-radius:8px;background:rgba(77,155,255,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128202;</span> ranked threats</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">scored, machine-readable, and diffed against the last run</div></div></div></div><div style="font-size:12px;color:#8A93A6;margin-top:8px;line-height:1.5">Read only the first column and you get a threat model that is identical for two deployments of the same repository — one behind a private load balancer with no egress, one not. It is wrong about both.</div>

## 4 · Stage 5 — derive threats from all six

In [ ]:
from collections import defaultdict

# --- input 1: the stage-4 architecture map ---------------------------------
ARCH = {
 "entry_points": [
   {"unit": "get_booking",    "component": "src/api", "auth": "session"},
   {"unit": "upload_voucher", "component": "src/api", "auth": "session"},
   {"unit": "health",         "component": "src/api", "auth": "none"},
 ],
 "flows": [("get_booking", "load_booking"), ("get_booking", "render"),
           ("upload_voucher", "store"), ("load_booking", "execute"),
           ("store", "open")],
 "sinks": [{"unit": "load_booking", "sink": "execute", "resource": "database"},
           {"unit": "store", "sink": "open", "resource": "voucher_bucket"}],
 "assets": {"database":       {"data": ("customer", "financial"), "value": 5},
            "voucher_bucket": {"data": ("documents",),            "value": 3}},
}

# --- inputs 2-6: what the rest of the estate already knows -----------------
CLOUD_POLICY = {           # security groups / ingress: is it on the internet?
 "get_booking":    {"exposed": "internet", "waf": True},
 "upload_voucher": {"exposed": "internet", "waf": False},
 "health":         {"exposed": "vpc-only", "waf": False},
}
CSPM = [                   # live posture findings, not code
 {"resource": "voucher_bucket", "finding": "bucket policy allows public read",
  "severity": 4},
]
ENTITLEMENTS = {           # what the running role may do
 "src/api": {"db:select", "db:update", "s3:GetObject", "s3:PutObject"},
}
IAM = {                    # who can become that role
 "src/api": {"assumable_by": ["ci-deploy-role"], "mfa_required": True},
}
EGRESS = {                 # NetworkPolicy / firewall: can anything leave?
 "src/api": {"default_deny": True, "allowed": ["db.prod:5432"]},
}

VECTOR_FOR = {
 "database":       [("CWE-89", "SQL injection", 5)],
 "voucher_bucket": [("CWE-22", "path traversal", 4),
                    ("CWE-434", "unrestricted upload", 4)],
}

def reachable(entry, flows):
    adj = defaultdict(list)
    for a, b in flows:
        adj[a].append(b)
    seen, stack = set(), [entry]
    while stack:
        for m in adj[stack.pop()]:
            if m not in seen:
                seen.add(m); stack.append(m)
    return seen

def threat_model(arch, cloud, cspm, entitlements, iam, egress):
    threats = []
    cspm_by_resource = defaultdict(int)
    for f in cspm:
        cspm_by_resource[f["resource"]] += f["severity"]
    for ep in arch["entry_points"]:
        reach = reachable(ep["unit"], arch["flows"])
        exposure = cloud.get(ep["unit"], {})
        for sink in arch["sinks"]:
            if sink["unit"] not in reach:
                continue
            asset = arch["assets"][sink["resource"]]
            comp = ep["component"]
            for cwe, name, base in VECTOR_FOR.get(sink["resource"], []):
                why, score = [], base + asset["value"]
                if ep["auth"] == "none":
                    score += 2; why.append("unauthenticated")
                # exposure: the single biggest correction the map cannot make
                if exposure.get("exposed") == "internet":
                    score += 2; why.append("internet-facing")
                else:
                    score -= 3; why.append("vpc-only")
                if exposure.get("exposed") == "internet" and not exposure.get("waf"):
                    score += 1; why.append("no WAF")
                score += cspm_by_resource[sink["resource"]]
                if cspm_by_resource[sink["resource"]]:
                    why.append("live CSPM finding")
                # entitlement: write beats read, and the role decides which
                if {"db:update", "s3:PutObject"} & entitlements.get(comp, set()):
                    score += 1; why.append("role holds write")
                if "*" in iam.get(comp, {}).get("assumable_by", []):
                    score += 2; why.append("role assumable by *")
                if not egress.get(comp, {}).get("default_deny", True):
                    score += 2; why.append("egress open")
                threats.append({
                  "entry": ep["unit"], "sink": sink["unit"], "cwe": cwe,
                  "vector": name, "score": score, "why": why,
                  "path": f"{ep['unit']} -> ... -> {sink['unit']}"})
    # score first, then a full tiebreak, so two machines agree
    return sorted(threats,
                  key=lambda t: (-t["score"], t["cwe"], t["entry"], t["sink"]))

TM = threat_model(ARCH, CLOUD_POLICY, CSPM, ENTITLEMENTS, IAM, EGRESS)
print(f"{'entry':16s}{'sink':14s}{'cwe':9s}{'score':>6}  why")
print("-" * 92)
for t in TM:
    print(f"{t['entry']:16s}{t['sink']:14s}{t['cwe']:9s}{t['score']:>6}  "
          f"{', '.join(t['why'])}")
print(f"\n{len(TM)} threats. Nobody wrote this; it fell out of six files that")
print("already existed in the estate.")

## 5 · The same code, two estates

This is the argument for reading past the map. Nothing below changes a line of the repository — only the configuration around it.

In [ ]:
# Same repository. Private load balancer, default-deny egress, no wildcard
# trust policy, and the CSPM finding remediated.
HARDENED = threat_model(
  ARCH,
  {k: {"exposed": "vpc-only", "waf": True} for k in CLOUD_POLICY},
  [],                                                        # CSPM clean
  {"src/api": {"db:select", "s3:GetObject"}},                # read-only role
  {"src/api": {"assumable_by": ["ci-deploy-role"], "mfa_required": True}},
  {"src/api": {"default_deny": True, "allowed": ["db.prod:5432"]}},
)

print(f"{'threat':38s}{'as deployed':>13}{'hardened':>11}")
print("-" * 62)
by_key = {(t["entry"], t["sink"], t["cwe"]): t["score"] for t in HARDENED}
for t in TM:
    k = (t["entry"], t["sink"], t["cwe"])
    print(f"{t['cwe'] + '  ' + t['path']:38s}{t['score']:>13}{by_key[k]:>11}")

print(f"\nmax severity  {max(t['score'] for t in TM)} -> "
      f"{max(t['score'] for t in HARDENED)}")
print()
print("Identical code. A model derived from the map alone would have scored")
print("these two deployments the same, and it would have been wrong about the")
print("first by understating it and wrong about the second by crying wolf.")
assert max(t["score"] for t in HARDENED) < max(t["score"] for t in TM)

## 6 · Where it breaks — the model that was true last quarter

Add one entry point. The hand-written threat model does not change, because documents do not change themselves.

In [ ]:
ARCH_V2 = {**ARCH,
 "entry_points": ARCH["entry_points"] + [
   {"unit": "admin_export", "component": "src/api", "auth": "none"}],
 "flows": ARCH["flows"] + [("admin_export", "load_booking"),
                           ("admin_export", "store")]}
CLOUD_V2 = {**CLOUD_POLICY,
            "admin_export": {"exposed": "internet", "waf": False}}

TM2 = threat_model(ARCH_V2, CLOUD_V2, CSPM, ENTITLEMENTS, IAM, EGRESS)

def diff(before, after):
    key = lambda t: (t["entry"], t["sink"], t["cwe"])
    b = {key(t): t for t in before}
    a = {key(t): t for t in after}
    # sorted() over a set difference is NOT deterministic across processes:
    # set iteration order depends on PYTHONHASHSEED, and a stable sort then
    # preserves that order for equal scores. Sort the keys first.
    return {"new": [a[k] for k in sorted(a.keys() - b.keys())],
            "removed": [b[k] for k in sorted(b.keys() - a.keys())],
            "max_before": max(t["score"] for t in before),
            "max_after": max(t["score"] for t in after)}

d = diff(TM, TM2)
print(f"threats before {len(TM)} -> after {len(TM2)}")
print(f"max severity   {d['max_before']} -> {d['max_after']}")
print("\nNEW THREATS:")
for t in sorted(d["new"], key=lambda t: (-t["score"], t["cwe"], t["entry"])):
    print(f"   [{t['score']:>2}] {t['cwe']:9s}{t['path']:38s}{', '.join(t['why'])}")
assert d["new"] and d["max_after"] >= d["max_before"]

## 7 · The control — and the gate that is not enough

Regenerate on every change to *any* input, and gate on the delta. The obvious gate counts new threats. Watch what it does with a pull request that adds none.

In [ ]:
def gate_new_only(before, after, critical_at=16):
    """The obvious gate: refuse a pull request that introduces a new critical."""
    d = diff(before, after)
    crit = [t for t in d["new"] if t["score"] >= critical_at]
    return not crit, {"new": len(d["new"]), "new_critical": len(crit)}

ok, info = gate_new_only(TM, TM2)
print(f"PR 1 - adds an unauthenticated handler   -> "
      f"{'PASS' if ok else 'FAIL'}  {info}")

# PR 2 touches no application code at all. It widens the IAM trust policy and
# removes the default-deny egress rule - two lines of terraform.
TM_TF = threat_model(ARCH, CLOUD_POLICY, CSPM, ENTITLEMENTS,
                     {"src/api": {"assumable_by": ["ci-deploy-role", "*"],
                                  "mfa_required": False}},
                     {"src/api": {"default_deny": False,
                                  "allowed": ["0.0.0.0/0"]}})
ok2, info2 = gate_new_only(TM, TM_TF)
print(f"PR 2 - two lines of terraform            -> "
      f"{'PASS' if ok2 else 'FAIL'}  {info2}")
print()
print("PR 2 introduced no new threat, so a gate that counts new threats waves")
print("it through. Every existing threat got worse:")
by_key = {(t["entry"], t["sink"], t["cwe"]): t["score"] for t in TM_TF}
for t in TM:
    k = (t["entry"], t["sink"], t["cwe"])
    print(f"   {t['cwe']:9s}{t['path']:38s}{t['score']:>3} -> {by_key[k]}")

## 8 · The gate that is

Count escalation as well as arrival. A threat that was medium and is now critical is a regression, and the pull request that caused it did not touch a line of application code.

In [ ]:
def threat_gate(before, after, critical_at=16, max_escalation=2):
    d = diff(before, after)
    prev = {(t["entry"], t["sink"], t["cwe"]): t["score"] for t in before}
    new_crit = [t for t in d["new"] if t["score"] >= critical_at]
    escalated = [t for t in after
                 if (k := (t["entry"], t["sink"], t["cwe"])) in prev
                 and t["score"] - prev[k] > max_escalation]
    return (not new_crit and not escalated,
            {"new_critical": len(new_crit), "escalated": len(escalated),
             "detail": [f"{t['cwe']} via {t['path']}: "
                        f"{prev[(t['entry'], t['sink'], t['cwe'])]} -> {t['score']}"
                        for t in escalated]})

for label, model in (("PR 1 - unauthenticated handler", TM2),
                     ("PR 2 - two lines of terraform ", TM_TF)):
    ok, info = threat_gate(TM, model)
    print(f"{label} -> {'PASS' if ok else 'FAIL'}")
    print(f"   new_critical={info['new_critical']}  escalated={info['escalated']}")
    for line in info["detail"]:
        print(f"      {line}")

# And the fix for PR 1: require a session and put it behind the WAF.
ARCH_FIXED = {**ARCH_V2, "entry_points": [
  {**e, "auth": "session"} if e["unit"] == "admin_export" else e
  for e in ARCH_V2["entry_points"]]}
CLOUD_FIXED = {**CLOUD_V2, "admin_export": {"exposed": "internet", "waf": True}}
ok_fixed, info_fixed = threat_gate(
    TM, threat_model(ARCH_FIXED, CLOUD_FIXED, CSPM, ENTITLEMENTS, IAM, EGRESS))
print(f"\nPR 1, after requiring auth and adding the WAF -> "
      f"{'PASS' if ok_fixed else 'FAIL'}  {info_fixed['new_critical']} critical")

print()
print("Nobody wrote a document. The gate compared generated models and refused")
print("two named regressions - and the one it would have missed is the one")
print("that changed no code, which is the majority of how estates get worse.")
assert gate_new_only(TM, TM_TF)[0]          # v1 waves the terraform PR through
assert not threat_gate(TM, TM_TF)[0]       # v2 does not
assert not threat_gate(TM, TM2)[0] and ok_fixed

## What you just proved

Six threats are derived from six static inputs, each carrying the reasons its score moved — internet-facing, no WAF, live CSPM finding, role holds write, role assumable by `*`, egress open. The same repository deployed behind a private load balancer with default-deny egress and a read-only role scores materially lower on every row. Adding an unauthenticated handler fails the CI gate, and so does a pull request that changes only terraform.

## Your turn

Take one service and write down where each of the six inputs lives — the repository, the CSPM console, the terraform, the IAM policy, the NetworkPolicy. If any of them is "in somebody's head", that is the input your threat model is currently guessing at, and the guess is always the optimistic one.

---

**Next → [B2.3 · Vulnerability auditing: three generations of SAST](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*